# 📚 Library Management System
**Python Assignment** – Mini Library Management System

**Features:** Add Books • Issue Book • Return Book • Search Book • Reports • File Handling (CSV)

**Concepts used:** Functions, Loops, Lists & Dictionaries, File Handling, Exception Handling, Menu-driven Program

**Data files created automatically:**
- `books.csv` – all books (ID, name, author, total quantity, available quantity)
- `issued_records.csv` – every issue/return record of students

## Step 1: Imports, file names and setup

In [1]:
import csv
import os
from datetime import date

BOOKS_FILE = "books.csv"
ISSUED_FILE = "issued_records.csv"

BOOK_FIELDS = ["book_id", "book_name", "author", "total_quantity", "available_quantity"]
ISSUE_FIELDS = ["student_name", "book_id", "book_name", "issue_date", "return_date", "status"]

print("Setup complete ✅")

Setup complete ✅


## Step 2: File Handling functions
Books are kept in a **dictionary** (`book_id -> book details`) and issue records in a **list of dictionaries**. Both are loaded from / saved to CSV files.

In [2]:
def init_files():
    """Create the CSV files with headers if they do not exist yet."""
    if not os.path.exists(BOOKS_FILE):
        with open(BOOKS_FILE, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=BOOK_FIELDS).writeheader()
    if not os.path.exists(ISSUED_FILE):
        with open(ISSUED_FILE, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=ISSUE_FIELDS).writeheader()


def load_books():
    """Read books.csv and return a dictionary {book_id: book_dict}."""
    books = {}
    try:
        with open(BOOKS_FILE, newline="") as f:
            for row in csv.DictReader(f):
                row["total_quantity"] = int(row["total_quantity"])
                row["available_quantity"] = int(row["available_quantity"])
                books[row["book_id"]] = row
    except FileNotFoundError:
        print("books.csv not found, starting with empty library.")
    except (ValueError, KeyError):
        print("books.csv has invalid data. Please check the file.")
    return books


def save_books(books):
    """Write the books dictionary back to books.csv."""
    try:
        with open(BOOKS_FILE, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=BOOK_FIELDS)
            writer.writeheader()
            writer.writerows(books.values())
    except IOError as e:
        print("Error saving books:", e)


def load_records():
    """Read issued_records.csv and return a list of dictionaries."""
    records = []
    try:
        with open(ISSUED_FILE, newline="") as f:
            records = list(csv.DictReader(f))
    except FileNotFoundError:
        print("issued_records.csv not found, starting with no records.")
    return records


def save_records(records):
    """Write the issue records list back to issued_records.csv."""
    try:
        with open(ISSUED_FILE, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=ISSUE_FIELDS)
            writer.writeheader()
            writer.writerows(records)
    except IOError as e:
        print("Error saving records:", e)


def print_books_table(book_list):
    """Display a list of books in a neat table."""
    if not book_list:
        print("No books to show.")
        return
    print(f"{'ID':<8}{'Book Name':<32}{'Author':<24}{'Total':<7}{'Available':<9}")
    print("-" * 80)
    for b in book_list:
        print(f"{b['book_id']:<8}{b['book_name']:<32}{b['author']:<24}"
              f"{b['total_quantity']:<7}{b['available_quantity']:<9}")

## Step 3: Add Book

In [3]:
def add_book(books, book_id, name, author, quantity):
    """Add a new book. If the ID already exists, increase its quantity instead."""
    if quantity <= 0:
        return "❌ Quantity must be greater than 0."
    if book_id in books:
        books[book_id]["total_quantity"] += quantity
        books[book_id]["available_quantity"] += quantity
        save_books(books)
        return f"ℹ️ Book ID {book_id} already exists. Added {quantity} more copies."
    books[book_id] = {
        "book_id": book_id,
        "book_name": name,
        "author": author,
        "total_quantity": quantity,
        "available_quantity": quantity,
    }
    save_books(books)
    return f"✅ Book '{name}' added successfully."

## Step 4: Issue Book
Asks student name and book ID, reduces available quantity and stores the issue record.

In [4]:
def issue_book(books, records, student, book_id):
    """Issue a book to a student."""
    if not student.strip():
        return "❌ Student name cannot be empty."
    if book_id not in books:
        return "❌ Book ID not found."
    book = books[book_id]
    if book["available_quantity"] <= 0:
        return f"❌ Sorry, '{book['book_name']}' is currently not available."
    # same student should not hold two copies of the same book
    for r in records:
        if (r["student_name"].lower() == student.lower()
                and r["book_id"] == book_id and r["status"] == "Issued"):
            return f"❌ {student} has already issued this book."
    book["available_quantity"] -= 1
    records.append({
        "student_name": student,
        "book_id": book_id,
        "book_name": book["book_name"],
        "issue_date": str(date.today()),
        "return_date": "",
        "status": "Issued",
    })
    save_books(books)
    save_records(records)
    return f"✅ '{book['book_name']}' issued to {student}."

## Step 5: Return Book

In [5]:
def return_book(books, records, student, book_id):
    """Return a book and update the quantity."""
    for r in records:
        if (r["student_name"].lower() == student.lower()
                and r["book_id"] == book_id and r["status"] == "Issued"):
            r["status"] = "Returned"
            r["return_date"] = str(date.today())
            if book_id in books:
                books[book_id]["available_quantity"] += 1
            save_books(books)
            save_records(records)
            return f"✅ '{r['book_name']}' returned by {student}. Thank you!"
    return "❌ No matching issued record found for this student and book ID."

## Step 6: Search Book
Search by **book name** or **author name** (case-insensitive, partial match works).

In [6]:
def search_books(books, keyword, by="name"):
    """Return list of books matching the keyword in name or author."""
    keyword = keyword.strip().lower()
    key = "book_name" if by == "name" else "author"
    return [b for b in books.values() if keyword in b[key].lower()]

## Step 7: Display Reports
Shows total books, issued books and available books, plus the currently issued list.

In [7]:
def show_reports(books, records):
    total_titles = len(books)
    total_copies = sum(b["total_quantity"] for b in books.values())
    available = sum(b["available_quantity"] for b in books.values())
    issued = total_copies - available

    print("=" * 45)
    print("            LIBRARY REPORT")
    print("=" * 45)
    print(f"Total book titles      : {total_titles}")
    print(f"Total books (copies)   : {total_copies}")
    print(f"Issued books           : {issued}")
    print(f"Available books        : {available}")
    print("=" * 45)

    current = [r for r in records if r["status"] == "Issued"]
    print("\nCurrently Issued Books:")
    if not current:
        print("  (none)")
    for r in current:
        print(f"  - {r['book_name']} (ID {r['book_id']}) -> {r['student_name']} "
              f"[issued on {r['issue_date']}]")

## Step 8: Menu-driven program
Run the **last cell** (`main()`) to use the interactive menu. Exception handling is used so wrong input does not crash the program.

In [8]:
def get_int(prompt):
    """Keep asking until the user enters a valid whole number."""
    while True:
        try:
            return int(input(prompt))
        except ValueError:
            print("⚠️ Please enter a valid number.")


def main():
    init_files()
    while True:
        print("\n===== LIBRARY MANAGEMENT SYSTEM =====")
        print("1. Add Book")
        print("2. Issue Book")
        print("3. Return Book")
        print("4. Search Book")
        print("5. Display Reports")
        print("6. Show All Books")
        print("7. Exit")
        choice = input("Enter your choice (1-7): ").strip()

        books = load_books()
        records = load_records()

        if choice == "1":
            book_id = input("Book ID: ").strip()
            name = input("Book Name: ").strip()
            author = input("Author: ").strip()
            quantity = get_int("Quantity Available: ")
            if not book_id or not name or not author:
                print("❌ All fields are required.")
            else:
                print(add_book(books, book_id, name, author, quantity))

        elif choice == "2":
            student = input("Student Name: ").strip()
            book_id = input("Book ID: ").strip()
            print(issue_book(books, records, student, book_id))

        elif choice == "3":
            student = input("Student Name: ").strip()
            book_id = input("Book ID: ").strip()
            print(return_book(books, records, student, book_id))

        elif choice == "4":
            by = input("Search by (1) Book Name  (2) Author: ").strip()
            keyword = input("Enter keyword: ")
            results = search_books(books, keyword, "name" if by == "1" else "author")
            print_books_table(results)

        elif choice == "5":
            show_reports(books, records)

        elif choice == "6":
            print_books_table(list(books.values()))

        elif choice == "7":
            print("Thank you for using the Library Management System. Goodbye! 👋")
            break
        else:
            print("⚠️ Invalid choice. Please select 1-7.")

## Demo (sample output for screenshots)
Ye cell menu ke bagair functions ko call karke sample output dikhata hai.
Is cell ko run karke output ka screenshot le sakte hain.

In [9]:
# Start fresh for the demo
for f in (BOOKS_FILE, ISSUED_FILE):
    if os.path.exists(f):
        os.remove(f)
init_files()

books = load_books()
records = load_records()

print("--- 1. ADD BOOKS ---")
print(add_book(books, "B001", "Python Crash Course", "Eric Matthes", 5))
print(add_book(books, "B002", "Automate the Boring Stuff", "Al Sweigart", 4))
print(add_book(books, "B003", "Clean Code", "Robert C. Martin", 3))
print(add_book(books, "B004", "Fluent Python", "Luciano Ramalho", 2))
print(add_book(books, "B005", "Learning Python", "Mark Lutz", 6))
print(add_book(books, "B006", "Python Cookbook", "David Beazley", 3))
print(add_book(books, "B003", "Clean Code", "Robert C. Martin", 2))   # duplicate ID
print(add_book(books, "B007", "Bad Book", "Nobody", 0))               # invalid quantity

print("\n--- 2. ISSUE BOOKS ---")
print(issue_book(books, records, "Ali Khan", "B001"))
print(issue_book(books, records, "Sara Ahmed", "B002"))
print(issue_book(books, records, "Bilal Raza", "B004"))
print(issue_book(books, records, "Ayesha Noor", "B004"))
print(issue_book(books, records, "Hamza Ali", "B004"))                # no copies left
print(issue_book(books, records, "Ali Khan", "B001"))                 # already issued
print(issue_book(books, records, "Zain", "B999"))                     # wrong ID

print("\n--- 3. RETURN BOOK ---")
print(return_book(books, records, "Sara Ahmed", "B002"))
print(return_book(books, records, "Sara Ahmed", "B002"))              # already returned

print("\n--- 4. SEARCH BOOKS ---")
print("Search by name 'python':")
print_books_table(search_books(books, "python", "name"))
print("\nSearch by author 'martin':")
print_books_table(search_books(books, "martin", "author"))

print("\n--- 5. REPORTS ---")
show_reports(books, records)

print("\n--- ALL BOOKS ---")
print_books_table(list(books.values()))

--- 1. ADD BOOKS ---
✅ Book 'Python Crash Course' added successfully.
✅ Book 'Automate the Boring Stuff' added successfully.
✅ Book 'Clean Code' added successfully.
✅ Book 'Fluent Python' added successfully.
✅ Book 'Learning Python' added successfully.
✅ Book 'Python Cookbook' added successfully.
ℹ️ Book ID B003 already exists. Added 2 more copies.
❌ Quantity must be greater than 0.

--- 2. ISSUE BOOKS ---
✅ 'Python Crash Course' issued to Ali Khan.
✅ 'Automate the Boring Stuff' issued to Sara Ahmed.
✅ 'Fluent Python' issued to Bilal Raza.
✅ 'Fluent Python' issued to Ayesha Noor.
❌ Sorry, 'Fluent Python' is currently not available.
❌ Ali Khan has already issued this book.
❌ Book ID not found.

--- 3. RETURN BOOK ---
✅ 'Automate the Boring Stuff' returned by Sara Ahmed. Thank you!
❌ No matching issued record found for this student and book ID.

--- 4. SEARCH BOOKS ---
Search by name 'python':
ID      Book Name                       Author                  Total  Available
--------------

## ▶️ Run the interactive menu
Is cell ko run karein aur menu use karein.

In [10]:
main()


===== LIBRARY MANAGEMENT SYSTEM =====
1. Add Book
2. Issue Book
3. Return Book
4. Search Book
5. Display Reports
6. Show All Books
7. Exit
✅ Book 'Python Masters' added successfully.

===== LIBRARY MANAGEMENT SYSTEM =====
1. Add Book
2. Issue Book
3. Return Book
4. Search Book
5. Display Reports
6. Show All Books
7. Exit
✅ Book 'Deep Learning' added successfully.

===== LIBRARY MANAGEMENT SYSTEM =====
1. Add Book
2. Issue Book
3. Return Book
4. Search Book
5. Display Reports
6. Show All Books
7. Exit
Thank you for using the Library Management System. Goodbye! 👋
